In [2]:
import polars as pl

In [3]:
itemcf_recall_path= '/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/itemcf_recall.parquet'
popularity_recall_week1_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week1.parquet'
popularity_recall_week2_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week2.parquet'
popularity_recall_week3_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week3.parquet'
popularity_recall_week4_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week4.parquet'
repurchase_recall_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/repurchase.parquet'
w2vec_recall_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/w2vec.parquet'

In [4]:
itemcf_recall = pl.read_parquet(itemcf_recall_path)

popularity_recall_week1 = pl.read_parquet(popularity_recall_week1_path)
popularity_recall_week2 = pl.read_parquet(popularity_recall_week2_path)
popularity_recall_week3 = pl.read_parquet(popularity_recall_week3_path)
popularity_recall_week4 = pl.read_parquet(popularity_recall_week4_path)

repurchase_recall = pl.read_parquet(repurchase_recall_path)

w2vec_recall = pl.read_parquet(w2vec_recall_path)

In [5]:
itemcf_recall.columns,w2vec_recall.columns

(['customer_id', 'article_id', 'score'],
 ['customer_id', 'article_id', 'score'])

In [6]:
itemcf_recall = itemcf_recall.rename({"score": "itemcf_score"})
w2vec_recall = w2vec_recall.rename({"score": "w2v_score"})

In [7]:
itemcf_recall.schema

Schema([('customer_id', String),
        ('article_id', Float64),
        ('itemcf_score', Float64)])

In [8]:
w2vec_recall.schema

Schema([('customer_id', String),
        ('article_id', Int64),
        ('w2v_score', Float64)])

In [9]:
repurchase_recall.schema

Schema([('customer_id', String), ('article_id', Int64), ('rank', UInt8)])

In [10]:
repurchase_recall.schema

Schema([('customer_id', String), ('article_id', Int64), ('rank', UInt8)])

In [11]:
itemcf_recall = itemcf_recall.with_columns(
    pl.col("article_id").cast(pl.Int64)
)

In [12]:
itemcf_recall = itemcf_recall.with_columns([
    pl.lit(1).alias("from_itemcf"),
])

w2vec_recall = w2vec_recall.with_columns([
    pl.lit(1).alias("from_w2vec"),
])

popularity_recall_week1 = popularity_recall_week1.with_columns([
    pl.lit(1).alias("from_popularity"),
])

popularity_recall_week2 = popularity_recall_week2.with_columns([
    pl.lit(1).alias("from_popularity"),
])

popularity_recall_week3 = popularity_recall_week3.with_columns([
    pl.lit(1).alias("from_popularity"),
])

popularity_recall_week4 = popularity_recall_week4.with_columns([
    pl.lit(1).alias("from_popularity"),
])

repurchase_recall = repurchase_recall.with_columns([
    pl.lit(1).alias("from_repurchase"),
])


In [13]:
popularity_recall_week1=popularity_recall_week1.drop(['rank'])
popularity_recall_week2=popularity_recall_week2.drop(['rank'])
popularity_recall_week3=popularity_recall_week3.drop(['rank'])
popularity_recall_week4=popularity_recall_week4.drop(['rank'])
repurchase_recall=repurchase_recall.drop(['rank'])

In [14]:
# 1. 把所有 recall 放到一个 list 里
dfs = [
    itemcf_recall,
    w2vec_recall,
    popularity_recall_week1,
    popularity_recall_week2,
    popularity_recall_week3,
    popularity_recall_week4,
    repurchase_recall,
]

In [15]:
fill_zero_cols = [
    "itemcf_score",
    "w2v_score",
    "from_itemcf",
    "from_w2vec",
    "from_popularity",
    "from_repurchase",
]

data =pl.concat(dfs, how="diagonal") # 默认会将缺失的部分补成null
for subdata in dfs:
    del subdata

In [ ]:
data.describe()

In [ ]:
data.to_pandas().info()

In [ ]:
data=data.with_columns([
        pl.col(c).fill_null(0)
        for c in fill_zero_cols
    ])